# ADS Bibliometric Analysis Pipeline

This notebook is the single entry point for the `ads_bib` package.
It executes all steps sequentially, with configuration cells between steps
so that downstream parameters can be set based on upstream results.

**Pipeline Steps:**
1. Search & Export â€” Query ADS API, resolve bibcodes to metadata
2. Translation â€” Detect languages, translate non-English titles/abstracts
3. Tokenization â€” Create full_text, tokenize with spaCy
4. AND â€” Author Name Disambiguation (placeholder)
5. Topic Modeling & Curation â€” BERTopic + datamapplot + cluster removal
6. Citation Networks â€” Direct, Co-Citation, Bibliographic Coupling, Author Co-Citation

---
## Setup

In [1]:
import os
from pathlib import Path

# Initialize paths (relative to this notebook's location)
from ads_bib.config import init_paths, load_env
from ads_bib.run_manager import RunManager
from ads_bib._utils.costs import CostTracker

paths = init_paths()  # uses CWD = notebook directory
load_env()

# Cost tracker for all API calls (OpenRouter, etc.)
tracker = CostTracker()

# Verify
print(f"Project root: {paths['project_root']}")
print(f"Data dir:     {paths['data']}")

# If you see a spaCy version warning later, update the model:
# !python -m spacy download en_core_web_lg

# --- INITIALIZE RUN ---
# This tracks all parameters and saves outputs to a unique folder
run = RunManager(run_name='ADS_Curation_Run')


Project root: c:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\ADS_Pipeline
Data dir:     c:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\ADS_Pipeline\data
Run initialized: run_20260221_145856_ADS_Curation_Run
All run outputs will be saved to: c:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\ADS_Pipeline\runs\run_20260221_145856_ADS_Curation_Run


---
## Global Run Control

Set run-level switches here. Phase-specific parameters are configured in each phase section below.


In [2]:
import random
import numpy as np

# --- RUN CONTROL ---
# 0 = Run everything (Search to End)
# 4 = Load processed data (Skip Search, Translate, Tokenize) -> Start at Topic Modeling
START_AT_PHASE = 0
FORCE_REFRESH = True

# Deterministic seed for notebook-side randomness (sampling + reduction params)
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Shared OpenRouter pricing mode (used in translation, embeddings, topic labeling)
OPENROUTER_COST_MODE = "hybrid"  # 'hybrid', 'strict', 'fast'


---
# Phase 1: Search & Export

Query the NASA ADS API for publications matching a search string,
then resolve all bibcodes (publications + references) into full metadata.

### 1.1 Search Configuration

Set query, years, and retrieval limits for your research question.
These values define the corpus scope for all downstream steps.


In [3]:
# # --- SEARCH CONFIGURATION ---
# # Modify the query string for your research question.
# # See: https://ui.adsabs.harvard.edu/help/search/search-syntax

# ADS_TOKEN = os.getenv("ADS_API_KEY")

# Set_A = "docs(library/P0hyiLw0T8qsyHSBpWigJA)"
# Set_B = f"citations({Set_A}) AND year:1915-2000"
# Set_C = f"citations(citations({Set_A})) AND year:1915-2000"
# Set_D = f"({Set_A}) OR ({Set_B}) OR ({Set_C})"
# Set_T = "(docs(library/_-RCjJotRKCZC1PP03MqwA) AND year:1915-2000)"

# gravity_relativity_terms = [
#     '(general AND relativi*)', 'gravit*', '(allgemein* AND relativi*)',
#     '"relativité générale"', '"teoria della relatività"',
#     '"Gravité quantique"', '"Gravità quantistica"',
#     '(einheitlich* AND feld*)', 'Quantengravitation', '"champ unifié"',
#     '(unified AND field*)', '"quantum gravity"', 'cosmo*', 'Kosmo*',
# ]
# Set_E = f"abs:({' OR '.join(gravity_relativity_terms)}) AND year:1915-2000"

# # Example quick focus query
# # QUERY = 'author:"Treder, H*"'
# # Alternative broader query:
# QUERY = f"({Set_D}) OR ({Set_T}) OR ({Set_E})"


In [4]:
# ...existing code...
# --- SEARCH CONFIGURATION ---
# Modify the query string for your research question.
# See: https://ui.adsabs.harvard.edu/help/search/search-syntax

ADS_TOKEN = os.getenv("ADS_API_KEY")

# 1. Historischer Seed: Einsteins Publikationen (1911-1955)
Set_A = "docs(library/P0hyiLw0T8qsyHSBpWigJA)"

# 2. Direkte Rezeption: 1-Hop Zitationen ab 1915
Set_B = f"citations({Set_A}) AND year:1915-2000"

# 3. Strukturelle Anker-Begriffe (Mehrsprachig: EN, DE, FR, IT)
generic_anchors = [
    'relativ*', 'einstein*', 
    'spacetime', '"space-time"', 'raumzeit', '"espace-temps"', 'spaziotempo', '"spazio-tempo"',
    'tensor*', 'tenseur*', 
    'metric*', 'metrik*', 'metriqu*', 
    'curvature', 'krümmung', 'kruemmung', 'courbure', 'curvatura',
    'geodesic*', 'geodät*', 'geodaet*', 'geodesiqu*', 'geodetic*'
]

# 4. Indirekte Rezeption: 2-Hop Zitationen ab 1915 (Gefiltert durch Anker)
Set_C = f"citations(citations({Set_A})) AND year:1915-2000 AND abs:({' OR '.join(generic_anchors)})"

# 5. Treder Library
Set_T = "(docs(library/_-RCjJotRKCZC1PP03MqwA) AND year:1915-2000)"

# 6. Text-Suche: Core-Begriffe und verankerte breite Begriffe
grg_core_terms = [
    '(general AND relativi*)', '(allgemein* AND relativi*)',
    '"relativité générale"', '"relatività generale"',
    '"Gravité quantique"', '"Gravità quantistica"', 'Quantengravitation', '"quantum gravity"',
    '(einheitlich* AND feld*)', '"champ unifié"', '(unified AND field*)'
]

broad_terms = ['gravit*', 'cosmo*', 'Kosmo*']
anchored_broad = f"({' OR '.join(broad_terms)}) AND ({' OR '.join(generic_anchors)})"

Set_E = f"abs:({' OR '.join(grg_core_terms)} OR ({anchored_broad})) AND year:1911-2000"

# Finale Query
QUERY = f"({Set_A}) OR ({Set_B}) OR ({Set_C}) OR ({Set_T}) OR ({Set_E})"
# ...existing code...

### 1.2 Execute Search

Run the ADS search and persist bibcodes/results to this run folder.
This creates a reproducible input snapshot for later phases.


In [5]:
if START_AT_PHASE <= 1:
    from ads_bib.search import search_ads, save_search_results
    from ads_bib._utils.io import load_pickle
    
    latest = paths["raw"] / "search_results_latest.pkl"
    
    if FORCE_REFRESH or not latest.exists():
        bibcodes, references, esources, fulltext_urls = search_ads(QUERY, ADS_TOKEN)
        save_search_results((bibcodes, references, esources, fulltext_urls), paths["raw"])
    else:
        print(f"Loading cached results: {latest.name}")
        bibcodes, references, esources, fulltext_urls = load_pickle(latest)
    
    print(f"\nBibcodes:        {len(bibcodes):,}")
    print(f"Unique refs:     {len(set(r for rl in references for r in rl)):,}")
    print(f"Total refs:      {sum(len(rl) for rl in references):,}")
else:
    print(f"Skipping Phase 1 Search (START_AT_PHASE={START_AT_PHASE})")

Starting ADS search ...
Server error 502. Retry 1/5 in 2s ...
  87,100 records fetched ...
Done. Total retrieved: 87,100
Saved: search_results_20260221_150320.pkl
Latest: search_results_latest.pkl

Bibcodes:        87,100
Unique refs:     307,885
Total refs:      1,108,054


### 1.3 Export & Resolve Metadata

Resolve publications and references to structured metadata tables.
This normalizes schemas before translation/tokenization/topic modeling.


In [6]:
if START_AT_PHASE <= 1:
    from ads_bib.export import resolve_dataset
    from ads_bib._utils.io import save_json_lines, load_json_lines
    
    pubs_path = paths["raw"] / "publications.json"
    refs_path = paths["raw"] / "references.json"
    
    if FORCE_REFRESH or not pubs_path.exists():
        publications, refs = resolve_dataset(
            bibcodes, references, esources, fulltext_urls, ADS_TOKEN
        )
        save_json_lines(publications, pubs_path)
        save_json_lines(refs, refs_path)
    else:
        print("Loading cached exports ...")
        publications = load_json_lines(pubs_path)
        refs = load_json_lines(refs_path)
    
    print(f"\nPublications: {len(publications):,}")
    print(f"References:   {len(refs):,}")
    publications.info()

=== Exporting publications ===
Exporting 87,100 bibcodes in 44 chunks (5 workers) ...


Exporting:  92%|█████████▏| 80000/87100 [02:11<00:15, 446.49it/s] 

Server error 502. Retry 1/5 in 2s ...


Exporting: 100%|██████████| 87100/87100 [02:30<00:00, 579.68it/s]


Publications: 87,095 records

=== Exporting references (307,885 unique) ===
Exporting 307,885 bibcodes in 154 chunks (5 workers) ...


Exporting: 100%|██████████| 307885/307885 [06:59<00:00, 734.19it/s] 


References: 307,878 records

Publications: 87,095
References:   307,878
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 87095 entries, 0 to 87094
Data columns (total 18 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   Bibcode               87095 non-null  object
 1   Author                87095 non-null  object
 2   Title                 87095 non-null  object
 3   Year                  87095 non-null  int32 
 4   Journal               87095 non-null  object
 5   Journal Abbreviation  87095 non-null  object
 6   Issue                 87095 non-null  object
 7   Volume                87095 non-null  object
 8   First Page            87095 non-null  object
 9   Last Page             87095 non-null  object
 10  Abstract              87095 non-null  object
 11  Keywords              87095 non-null  object
 12  DOI                   87095 non-null  object
 13  Affiliation           87095 non-null  object
 14  Category      

In [7]:
if START_AT_PHASE <= 1:
    display(publications.head(50))

,Bibcode,Author,Title,Year,Journal,Journal Abbreviation,Issue,Volume,First Page,Last Page,Abstract,Keywords,DOI,Affiliation,Category,Citation Count,References,PDF_URL
0,2000A&A...363.1081C,"[Claret, A.]",A new non-linear limb-darkening law for LTE st...,2000,Astronomy and Astrophysics,A&A,,363,1081,1190,"A new non-linear limb-darkening law, based on ...","STARS: ATMOSPHERES, STARS: BINARIES: ECLIPSING",,AA(Institute of Astrophysics of Andalusia),article,1118,"[1921MNRAS..81..361M, 1965BAICz..16..195G, 197...",https://ui.adsabs.harvard.edu/link_gateway/200...
1,2000PhRvL..85.5042P,"[Parikh, Maulik K., Wilczek, Frank]",Hawking Radiation As Tunneling,2000,Physical Review Letters,PhRvL,24,85,5042,5045,We present a short and direct derivation of Ha...,"High Energy Physics - Theory, General Relativi...",10.1103/PhysRevLett.85.5042,"AA(Princeton University, Department of Physics...",article,1810,"[1921CR....173..677P, 1975CMaPh..43..199H, 197...",None
2,2000PhRvL..85.4438A,"[ArmendarizPicon, C., Mukhanov, V., Steinhardt...",Dynamical Solution to the Problem of a Small C...,2000,Physical Review Letters,PhRvL,21,85,4438,4441,Increasing evidence suggests that most of the ...,"Astrophysics, General Relativity and Quantum C...",10.1103/PhysRevLett.85.4438,"AA(University of Munich, Department of Physics...",article,1898,"[1986NuPhB.277....1G, 1988ApJ...325L..17P, 198...",None
3,2000ApJ...542..281A,"[Alcock, C., Allsman, R. A., Alves, D. R., Axe...",The MACHO Project: Microlensing Results from 5...,2000,The Astrophysical Journal,ApJ,1,542,281,307,We report on our search for microlensing towar...,"Cosmology: Dark Matter, Galaxy: Halo, Galaxy: ...",10.1086/309512,"AA(Lawrence Livermore National Laboratory, Cal...",article,937,"[1964MNRAS.128..295R, 1986ApJ...304....1P, 199...",None
4,2000PhRvD..62h4003V,"[Virbhadra, K. S., Ellis, George F. R.]",Schwarzschild black hole lensing,2000,Physical Review D,PhRvD,8,62,084003,,We study strong gravitational lensing due to a...,"04.70.Bw, 04.80.Cc, 97.60.Lf, 98.62.Sb, Classi...",10.1103/PhysRevD.62.084003,"AA(University of Cape Town, South Africa); AB(...",article,1111,"[1964MNRAS.128..295R, 1964PhRv..133..835L, 197...",None
5,2000PhLB..493..149A,"[AyónBeato, E., García, A.]",The Bardeen model as a nonlinear magnetic mono...,2000,Physics Letters B,PhLB,1,493,149,152,The Bardeen model - the first regular black ho...,"General Relativity and Quantum Cosmology, High...",10.1016/S0370-2693(00)01125-4,AA(Center for Research and Advanced Studies of...,article,843,"[1996CQGra..13L..51M, 1996PhRvD..53.3215B, 199...",None
6,2000PhRvD..62f4015B,"[Buonanno, Alessandra, Damour, Thibault]",Transition from inspiral to plunge in binary b...,2000,Physical Review D,PhRvD,6,62,064015,,Combining recent techniques giving nonperturba...,"04.30.-w, 04.25.Nx, Gravitational waves: theor...",10.1103/PhysRevD.62.064015,"AA(California Institute of Technology, Divisio...",article,780,"[1960PhRv..120..313A, 1971ApJ...170L.105P, 197...",None
7,2000PhRvL..85.2236B,"[Boisseau, B., EspositoFarèse, G., Polarski, D...",Reconstruction of a Scalar-Tensor Theory of Gr...,2000,Physical Review Letters,PhRvL,11,85,2236,2239,The present acceleration of the Universe stron...,"General Relativity and Quantum Cosmology, Astr...",10.1103/PhysRevLett.85.2236,AA(Laboratoire de Mathematiques et Physique Th...,article,808,"[1968IJTP....1...25B, 1970ApJ...161.1059N, 197...",None
8,2000PhRvD..63b3506G,"[Gordon, Christopher, Wands, David, Bassett, B...",Adiabatic and entropy perturbations from infla...,2000,Physical Review D,PhRvD,2,63,023506,,We study adiabatic (curvature) and entropy (is...,"98.80.Cq, Particle-theory and field-theory mod...",10.1103/PhysRevD.63.023506,"AA(University of Portsmouth, Institute of Cosm...",article,738,"[1974RSPSA.341...49S, 1980JETP...52..807L, 198...",None
9,2000ApJ...542..535G,"[Gnedin, Nickolay Y.]",Effect of Reionization on Structure Formation ...,2000,The Astrophysical Journal,ApJ,2,542,535,541,I use simulations of cosmo

---
# Phase 2: Translation

Detect non-English titles and abstracts with fasttext,
then translate them using either OpenRouter (API) or a local HuggingFace model.

### 2.1 Translation Configuration

Choose provider/model and translation target language.
Keep settings explicit so costs and outputs are reproducible.


In [8]:
# --- TRANSLATION CONFIGURATION ---
# Provider options:
# - 'openrouter' (API; requires OPENROUTER_API_KEY)
# - 'huggingface' (local transformers model)

TRANSLATION_PROVIDER = "openrouter"  # or "huggingface"
TRANSLATION_MODEL = "google/gemini-3-flash-preview"
TRANSLATION_API_KEY = os.getenv("OPENROUTER_API_KEY")
TRANSLATION_MAX_WORKERS = 10

# Path to fasttext language ID model:
# https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin
FASTTEXT_MODEL = paths["models"] / "lid.176.bin"


### 2.2 Language Detection

Detect language tags for configured text columns.
Only non-target-language rows are translated in the next step.


In [9]:
if START_AT_PHASE <= 2:
    from ads_bib.translate import detect_languages
    
    print("=== Publications ===")
    publications = detect_languages(publications, ["Title", "Abstract"], model_path=FASTTEXT_MODEL)
    
    print("\n=== References ===")
    refs = detect_languages(refs, ["Title", "Abstract"], model_path=FASTTEXT_MODEL)
else:
    print(f"Skipping Phase 2 Translation (START_AT_PHASE={START_AT_PHASE})")

=== Publications ===
  Title: 1,415 non-English entries detected
  Abstract: 428 non-English entries detected

=== References ===
  Title: 5,516 non-English entries detected
  Abstract: 785 non-English entries detected


### 2.3 Translate Non-English Entries

Translate non-English rows and track token/cost metadata.
This harmonizes text fields for downstream NLP and topic modeling.


In [10]:
if START_AT_PHASE <= 2:
    if TRANSLATION_PROVIDER not in {"openrouter", "huggingface"}:
        raise ValueError("TRANSLATION_PROVIDER must be 'openrouter' or 'huggingface'.")
    if TRANSLATION_PROVIDER == "openrouter" and not TRANSLATION_API_KEY:
        raise ValueError("OPENROUTER_API_KEY is required when TRANSLATION_PROVIDER='openrouter'.")
    if TRANSLATION_PROVIDER == "huggingface":
        try:
            import transformers  # noqa: F401
        except Exception as exc:
            raise ImportError(
                "TRANSLATION_PROVIDER='huggingface' requires 'transformers', 'torch', and 'accelerate'."
            ) from exc

    from ads_bib.translate import translate_dataframe

    print("=== Translating Publications ===")
    publications, cost_pubs = translate_dataframe(
        publications, ["Title", "Abstract"],
        provider=TRANSLATION_PROVIDER,
        model=TRANSLATION_MODEL,
        api_key=TRANSLATION_API_KEY,
        max_workers=TRANSLATION_MAX_WORKERS,
        openrouter_cost_mode=OPENROUTER_COST_MODE,
        cost_tracker=tracker,
    )

    print("\n=== Translating References ===")
    refs, cost_refs = translate_dataframe(
        refs, ["Title", "Abstract"],
        provider=TRANSLATION_PROVIDER,
        model=TRANSLATION_MODEL,
        api_key=TRANSLATION_API_KEY,
        max_workers=TRANSLATION_MAX_WORKERS,
        openrouter_cost_mode=OPENROUTER_COST_MODE,
        cost_tracker=tracker,
    )

    pub_cost = cost_pubs.get("cost_usd")
    ref_cost = cost_refs.get("cost_usd")
    print("\nTranslation Cost Snapshot:")
    print(f"  Publications: {'n/a' if pub_cost is None else f'${pub_cost:.4f}'}")
    print(f"  References:   {'n/a' if ref_cost is None else f'${ref_cost:.4f}'}")

else:
    print(f"Skipping Phase 2 Translation (START_AT_PHASE={START_AT_PHASE})")


=== Translating Publications ===
  Title: translating 1,415 entries with openrouter/google/gemini-3-flash-preview ...


  Title: 100%|██████████| 1415/1415 [03:53<00:00,  6.05it/s]


  Abstract: translating 428 entries with openrouter/google/gemini-3-flash-preview ...


  Abstract: 100%|██████████| 428/428 [02:26<00:00,  2.92it/s]


  Total translation cost: $0.3979 (1843/1843 priced; direct=1843, fetched=0, mode=hybrid)

=== Translating References ===
  Title: translating 5,516 entries with openrouter/google/gemini-3-flash-preview ...


  Title: 100%|██████████| 5516/5516 [15:58<00:00,  5.76it/s]  


  Abstract: translating 785 entries with openrouter/google/gemini-3-flash-preview ...


  Abstract: 100%|██████████| 785/785 [03:56<00:00,  3.32it/s]

  Total translation cost: $0.9687 (6301/6301 priced; direct=6301, fetched=0, mode=hybrid)

Translation Cost Snapshot:
  Publications: $0.3979
  References:   $0.9687


---
# Phase 3: Tokenization

Create `full_text` (Title + Abstract) and tokenize with spaCy.

In [11]:
if START_AT_PHASE <= 3:
    from ads_bib.tokenize import tokenize_texts
    
    publications = tokenize_texts(publications)
    refs = tokenize_texts(refs)
else:
    print(f"Skipping Phase 3 Tokenization (START_AT_PHASE={START_AT_PHASE})")

Tokenizing 87,095 documents with en_core_web_lg ...
  Done.
Tokenizing 307,878 documents with en_core_web_lg ...


KeyboardInterrupt: 

### Checkpoint: Save Phase 1-3 Results

Persist translated/tokenized data so topic modeling can restart from Phase 4.
This avoids rerunning API-heavy earlier phases.


In [12]:
if START_AT_PHASE <= 3:
    from ads_bib._utils.io import save_json_lines
    import shutil

    # 1. Save to global cache so next time we can START_AT_PHASE = 4
    global_pub = paths["cache"] / "publications_translated_tokenized.json"
    global_ref = paths["cache"] / "references_translated_tokenized.json"
    
    save_json_lines(publications, global_pub)
    save_json_lines(refs, global_ref)
    
    # 2. Copy a static snapshot to the current Run folder
    shutil.copy(global_pub, run.paths["data"] / "publications_translated_tokenized.json")
    shutil.copy(global_ref, run.paths["data"] / "references_translated_tokenized.json")
    print("Checkpoint saved to global cache and local run folder.")


Checkpoint saved to global cache and local run folder.


---
# Phase 4: Author Name Disambiguation (Placeholder)

This step will use an external AND package once it's ready.
For now, the pipeline continues without disambiguation.

In [ ]:
# === AND PLACEHOLDER ===
# Uncomment when the AND package is available:
#
# from and_package import disambiguate
# publications = disambiguate(publications)
# refs = disambiguate(refs)

print("AND step skipped (placeholder). Continuing without disambiguation.")

---
# Phase 5: Topic Modeling & Curation

Use BERTopic to discover thematic clusters, visualize with datamapplot,
then remove unwanted clusters to curate the dataset.

**Important:** Set parameters below based on your dataset size from Phase 1.

### 5.1 Embedding Configuration

Configure embedding provider/model and optional sampling.
These settings strongly affect topic quality and runtime/cost.


In [ ]:
# --- EMBEDDING CONFIGURATION ---
# Providers:
# - 'openrouter' (API; requires OPENROUTER_API_KEY)
# - 'huggingface_api' (remote API via LiteLLM)
# - 'local' (SentenceTransformers on local hardware)

EMBEDDING_PROVIDER = "openrouter"
EMBEDDING_MODEL = "google/gemini-embedding-001"
EMBEDDING_API_KEY = os.getenv("OPENROUTER_API_KEY")
EMBEDDING_MAX_WORKERS = 8  # Used for provider='openrouter'.

# For testing: set SAMPLE_SIZE to an integer (e.g. 200). None = full dataset.
SAMPLE_SIZE = None


### 5.2 Compute Embeddings

Create or load cached embeddings for `full_text`.
Caching keeps repeated runs fast and reproducible.


In [ ]:
from ads_bib._utils.io import load_json_lines
import shutil

if START_AT_PHASE >= 4:
    # We skipped the earlier phases, so we must load the data from global cache
    global_pub = paths["cache"] / "publications_translated_tokenized.json"
    global_ref = paths["cache"] / "references_translated_tokenized.json"
    
    if not global_pub.exists():
        raise FileNotFoundError(f"Cannot skip to Phase 4. Global cache missing: {global_pub}")
        
    print("Loading data from global cache (Phase 1-3 skipped) ...")
    publications = load_json_lines(global_pub)
    refs = load_json_lines(global_ref)
    
    # Snapshot into current run for provenance
    shutil.copy(global_pub, run.paths["data"] / "publications_translated_tokenized.json")
    shutil.copy(global_ref, run.paths["data"] / "references_translated_tokenized.json")

print(f"Loaded {len(publications):,} publications")
print(f"Loaded {len(refs):,} references")


In [ ]:
from ads_bib.topic_model import compute_embeddings

df = publications.copy()
if SAMPLE_SIZE is not None:
    df = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=RANDOM_SEED).reset_index(drop=True)
    print(f"SAMPLING: {len(df):,} documents")

if EMBEDDING_PROVIDER not in {"local", "huggingface_api", "openrouter"}:
    raise ValueError("EMBEDDING_PROVIDER must be 'local', 'huggingface_api', or 'openrouter'.")
if EMBEDDING_PROVIDER == "openrouter" and not EMBEDDING_API_KEY:
    raise ValueError("OPENROUTER_API_KEY is required when EMBEDDING_PROVIDER='openrouter'.")
if EMBEDDING_PROVIDER == "local":
    try:
        import sentence_transformers  # noqa: F401
    except Exception as exc:
        raise ImportError("EMBEDDING_PROVIDER='local' requires 'sentence-transformers'.") from exc
if EMBEDDING_PROVIDER in {"openrouter", "huggingface_api"}:
    try:
        import litellm  # noqa: F401
    except Exception as exc:
        raise ImportError("Remote embedding providers require 'litellm'.") from exc

documents = df["full_text"].tolist()

embeddings = compute_embeddings(
    documents,
    provider=EMBEDDING_PROVIDER,
    model=EMBEDDING_MODEL,
    cache_dir=paths["embeddings_cache"],
    max_workers=EMBEDDING_MAX_WORKERS,
    api_key=EMBEDDING_API_KEY,
    openrouter_cost_mode=OPENROUTER_COST_MODE,
    cost_tracker=tracker,
)
print(f"Embeddings shape: {embeddings.shape}")


### 5.3 Dimensionality Reduction Configuration

- Methods: `pacmap` (fast) or `umap` (preferred for hierarchical Toponymy structure).
- Heuristic: smaller/sparser sets use `n_neighbors` ~15; larger/denser sets often benefit from 50-60.
- Use `min_dist=0.0` for 5D clustering vectors and `min_dist=0.1` for 2D visualization.


In [ ]:
# --- DIMENSIONALITY REDUCTION ---
DIM_REDUCTION_METHOD = "umap"  # 'pacmap' or 'umap'

PARAMS_5D = dict(n_neighbors=15, min_dist=0.0, metric="cosine", random_state=RANDOM_SEED)
PARAMS_2D = dict(n_neighbors=15, min_dist=0.1, metric="cosine", random_state=RANDOM_SEED)

model_safe = EMBEDDING_MODEL.replace('/', '_')
CACHE_SUFFIX = f"{EMBEDDING_PROVIDER}_{model_safe}_{DIM_REDUCTION_METHOD}"


In [ ]:
from ads_bib.topic_model import reduce_dimensions

reduced_5d, reduced_2d = reduce_dimensions(
    embeddings,
    method=DIM_REDUCTION_METHOD,
    params_5d=PARAMS_5D,
    params_2d=PARAMS_2D,
    random_state=RANDOM_SEED,
    cache_dir=paths["dim_reduction_cache"],
    cache_suffix=CACHE_SUFFIX,
)

### 5.4 Clustering Configuration

**Adjust clustering granularity based on dataset size.**
- `MIN_CLUSTER_SIZE` controls BERTopic/HDBSCAN topic granularity (roughly ~0.1% of docs).
- `BASE_MIN_CLUSTER_SIZE` controls Toponymy micro-cluster granularity.
- `TOPONYMY_LAYER_INDEX` selects the fallback primary layer for visualization.


In [ ]:
# --- CLUSTERING CONFIGURATION ---
CLUSTERING_METHOD = "fast_hdbscan"  # or 'hdbscan'

n_docs = len(df)
MIN_CLUSTER_SIZE = max(10, int(n_docs * 0.001))
BASE_MIN_CLUSTER_SIZE = max(10, int(n_docs * 0.001))
print(f"Dataset: {n_docs:,} documents")
print(f"  MIN_CLUSTER_SIZE={MIN_CLUSTER_SIZE}")
print(f"  BASE_MIN_CLUSTER_SIZE={BASE_MIN_CLUSTER_SIZE}")

CLUSTER_PARAMS = dict(
    min_cluster_size=MIN_CLUSTER_SIZE,
    min_samples=3,
    cluster_selection_method="eom",
    cluster_selection_epsilon=0.02,
)

TOPONYMY_CLUSTER_PARAMS = dict(
    min_clusters=5,
    min_samples=3,
    base_min_cluster_size=BASE_MIN_CLUSTER_SIZE,
)

# EVoC uses a backend-specific constructor in some versions.
# Keep this dict conservative; unsupported keys are filtered in fit_toponymy.
TOPONYMY_EVOC_CLUSTER_PARAMS = dict(
    min_clusters=5,
    min_samples=3,
    base_min_cluster_size=BASE_MIN_CLUSTER_SIZE,
)

TOPONYMY_LAYER_INDEX = 0


### 5.5 Backend & LLM Configuration

Backend matrix:
- `bertopic`: BERTopic on 5D reduced vectors + optional outlier reassignment refresh.
- `toponymy`: Toponymy + `ToponymyClusterer` on 5D reduced vectors.
- `toponymy_evoc`: Toponymy + `EVoCClusterer` on raw embeddings (no 5D clustering vectors).

LLM provider matrix:
- BERTopic: `local`, `huggingface_api`, `openrouter`
- Toponymy / Toponymy+EVoC: `local` (Toponymy HuggingFaceNamer) or `openrouter`

`MIN_DF` guidance:
- small sample runs: often `1`
- full runs: usually `2+`


In [ ]:
# --- TOPIC BACKEND + LLM LABELING ---
TOPIC_BACKEND = "toponymy_evoc"  # 'bertopic', 'toponymy', 'toponymy_evoc'

# BERTopic providers: 'local', 'huggingface_api', 'openrouter'
# Toponymy providers: 'local' (HuggingFaceNamer) or 'openrouter'
LLM_PROVIDER = "openrouter"
LLM_MODEL = "google/gemini-3-flash-preview"
LLM_API_KEY = os.getenv("OPENROUTER_API_KEY")

PIPELINE_MODELS = ["POS", "KeyBERT", "MMR"]
PARALLEL_MODELS = ["MMR", "POS", "KeyBERT"]

TOPONYMY_EMBEDDING_MODEL = EMBEDDING_MODEL
MIN_DF = 1 if (SAMPLE_SIZE and SAMPLE_SIZE < 1000) else 2


In [ ]:
from ads_bib.topic_model import fit_bertopic, fit_toponymy, reduce_outliers, build_topic_dataframe
import numpy as np

valid_backends = {"bertopic", "toponymy", "toponymy_evoc"}
if TOPIC_BACKEND not in valid_backends:
    raise ValueError(
        f"Invalid TOPIC_BACKEND '{TOPIC_BACKEND}'. Use 'bertopic', 'toponymy', or 'toponymy_evoc'."
    )

allowed_llm = (
    {"local", "huggingface_api", "openrouter"}
    if TOPIC_BACKEND == "bertopic"
    else {"local", "openrouter"}
)
if LLM_PROVIDER not in allowed_llm:
    raise ValueError(
        f"LLM_PROVIDER '{LLM_PROVIDER}' is not valid for TOPIC_BACKEND='{TOPIC_BACKEND}'. "
        f"Allowed: {sorted(allowed_llm)}"
    )
if LLM_PROVIDER == "openrouter" and not LLM_API_KEY:
    raise ValueError("OPENROUTER_API_KEY is required when LLM_PROVIDER='openrouter'.")

if LLM_PROVIDER == "local":
    if TOPIC_BACKEND == "bertopic":
        try:
            import transformers  # noqa: F401
        except Exception as exc:
            raise ImportError("BERTopic local labeling requires 'transformers'.") from exc
    else:
        try:
            import sentence_transformers  # noqa: F401
            from toponymy.llm_wrappers import HuggingFaceNamer  # noqa: F401
        except Exception as exc:
            raise ImportError(
                "Toponymy local labeling requires 'sentence-transformers' and Toponymy HuggingFaceNamer support."
            ) from exc

selected_toponymy_cluster_params = (
    TOPONYMY_EVOC_CLUSTER_PARAMS
    if TOPIC_BACKEND == "toponymy_evoc"
    else TOPONYMY_CLUSTER_PARAMS
)

if TOPIC_BACKEND == "bertopic":
    topic_model = fit_bertopic(
        documents,
        reduced_5d,
        llm_provider=LLM_PROVIDER,
        llm_model=LLM_MODEL,
        pipeline_models=PIPELINE_MODELS,
        parallel_models=PARALLEL_MODELS,
        min_df=MIN_DF,
        clustering_method=CLUSTERING_METHOD,
        clustering_params=CLUSTER_PARAMS,
        api_key=LLM_API_KEY,
        openrouter_cost_mode=OPENROUTER_COST_MODE,
        cost_tracker=tracker,
    )

    topics = np.array(topic_model.topics_)
    topics = reduce_outliers(
        topic_model,
        documents,
        topics,
        reduced_5d,
        threshold=0.8,
        llm_provider=LLM_PROVIDER,
        llm_model=LLM_MODEL,
        api_key=LLM_API_KEY,
        openrouter_cost_mode=OPENROUTER_COST_MODE,
        cost_tracker=tracker,
    )
    topic_info = topic_model.get_topic_info()

elif TOPIC_BACKEND in {"toponymy", "toponymy_evoc"}:
    topic_model, topics, topic_info = fit_toponymy(
        documents,
        embeddings,
        reduced_5d,
        backend=TOPIC_BACKEND,
        layer_index=TOPONYMY_LAYER_INDEX,
        llm_provider=LLM_PROVIDER,
        llm_model=LLM_MODEL,
        embedding_model=TOPONYMY_EMBEDDING_MODEL,
        api_key=LLM_API_KEY,
        openrouter_cost_mode=OPENROUTER_COST_MODE,
        clusterer_params=selected_toponymy_cluster_params,
        cost_tracker=tracker,
    )

else:
    raise ValueError(
        f"Invalid TOPIC_BACKEND '{TOPIC_BACKEND}'. "
        "Use 'bertopic', 'toponymy', or 'toponymy_evoc'."
    )

df = build_topic_dataframe(
    df,
    topic_model,
    topics,
    reduced_2d,
    embeddings,
    topic_info=topic_info,
)
topic_info


### 5.6 Visualize Topics

Render an interactive topic map from 2D coordinates and topic labels.
Use this view to inspect cluster semantics before curation.


In [ ]:
from ads_bib.visualize import create_topic_map

# Use all layers for Toponymy, or just 'Name' for BERTopic
if TOPIC_BACKEND in {"toponymy", "toponymy_evoc"}:
    num_layers = len([col for col in df.columns if col.startswith("Topic_Layer_")])
    label_column = [f"Topic_Layer_{i}" for i in range(num_layers)]
else:
    label_column = "Name"

plot = create_topic_map(
    df,
    label_column=label_column,
    title="ADS Bibliometric Map",
    subtitle=f"Topics labeled with {LLM_PROVIDER}/{LLM_MODEL}",
    dark_mode=True,
    output_path=run.paths["plots"] / "topic_map.html",
)
# plot  # uncomment to display inline

### 5.7 Curate Dataset

Review the cluster summary, then remove clusters that don't fit your research question.

In [ ]:
from ads_bib.curate import get_cluster_summary, remove_clusters

get_cluster_summary(df)

### Checkpoint: Save Curated Dataset

Save the curated topic dataset as the handoff for citation analysis.
This provides a stable artifact for Phase 6 and external reuse.


In [ ]:
from ads_bib._utils.io import save_parquet

# Ensure References is list type for Parquet
df["References"] = df["References"].apply(lambda x: x if isinstance(x, list) else [])

save_parquet(df, run.paths["data"] / "curated_dataset.parquet")
print(f"Curated dataset: {len(df):,} documents")

In [ ]:
from ads_bib._utils.io import load_parquet

df = load_parquet(run.paths["data"] / "curated_dataset.parquet")
df.head(10)

---
# Phase 6: Citation Networks

Compute citation networks from the curated dataset and export
to GEXF (Gephi), Graphology JSON (Sigma.js), CSV, and/or Web of Science format.

### 6.1 Citation Configuration

Set network metrics, thresholds, filters, and export formats.
These parameters define which citation structures are constructed.


In [ ]:
# --- CITATION CONFIGURATION ---
CITE_METRICS = ["direct", "co_citation", "bibliographic_coupling", "author_co_citation"]
MIN_COUNTS = {
    "direct": 1,
    "co_citation": 1,
    "bibliographic_coupling": 1,
    "author_co_citation": 1,
}
AUTHORS_FILTER = None
OUTPUT_FORMAT = "gexf"  # 'gexf', 'graphology', 'csv', 'all'
CLUSTERS_TO_REMOVE = []

# Snapshot the full configuration once all phase configs are defined.
run.save_config(locals())


### 6.2 Build Node Table & Compute Networks

Build node metadata and compute selected citation networks.
Outputs are exported for Gephi/Sigma/CSV workflows.


In [ ]:
from ads_bib.citations import build_all_nodes, process_all_citations, export_wos_format

# Reload processed data if needed
all_nodes = build_all_nodes(publications, refs)

results = process_all_citations(
    bibcodes, references, publications, refs, all_nodes,
    metrics=CITE_METRICS,
    min_counts=MIN_COUNTS,
    authors_filter=AUTHORS_FILTER,
    output_format=OUTPUT_FORMAT,
    output_dir=run.paths["data"],
)

### 6.3 Export Web of Science Format

Export the curated corpus in WOS text format.
This supports downstream tooling such as CiteSpace/VOSviewer.


In [ ]:
suffix = "_filtered" if AUTHORS_FILTER else ""
export_wos_format(
    publications, refs,
    output_path=run.paths["data"] / f"download_wos_export{suffix}.txt",
)

---
# Summary

Final dataset statistics and output file listing.

In [ ]:
import os

print("="*60)
print("PIPELINE COMPLETE")
print("="*60)
print(f"Publications:     {len(publications):,}")
print(f"References:       {len(refs):,}")
print(f"Curated dataset:  {len(df):,}")
print(f"Topics found:     {df['topic_id'].nunique()}")
print()
print("Output files:")
for root, dirs, files in os.walk(run.paths["root"]):
    for f in sorted(files):
        fpath = Path(root) / f
        size_mb = fpath.stat().st_size / 1024 / 1024
        print(f"  {fpath.relative_to(run.paths['root'])} ({size_mb:.1f} MB)")

print()
print(tracker.compact_summary())
